# Dataset Preprocessing for Multimodal Lily (Phase 1 Projector Alignment)

This notebook compiles, resizes, tokenizes, and packages the dataset mix for Phase 1 projector pretraining. 
It loads all images and tokenized features in-memory before creating a Hugging Face Dataset from the list. Designed for environments with sufficient system RAM (e.g. Modal).

In [1]:
# Install dependencies
%uv pip install -q transformers datasets pillow huggingface_hub tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import random
import io
import torch
from PIL import Image
from tqdm.auto import tqdm
from datasets import Dataset, Features, Image as HFImage, Sequence, Value, load_dataset
from transformers import AutoTokenizer, AutoProcessor
from huggingface_hub import login

# HF Authentication
HF_TOKEN = os.environ.get("HF_TOKEN")
login(token=HF_TOKEN, add_to_git_credential=False)
print(">> Authenticated successfully with Hugging Face")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


>> Authenticated successfully with Hugging Face


In [3]:
# --- Model and Dataset Config ---
LLM_MODEL_ID = "abhinav0231/Lily-1.5b-v0.3"
VISION_MODEL_ID = "google/siglip2-so400m-patch14-384"

# Dataset mix targets (Dropped LAION to 100k as approved for efficiency & concentration)
NUM_LAION = 100_000
NUM_AI2D = 15_000         # lmms-lab/ai2d test split has ~3,088 samples
NUM_CHARTQA = 10_000      # HuggingFaceM4/ChartQA has 10,000 samples
NUM_DOCVQA = 5_350        # lmms-lab/DocVQA validation split has 5,349 samples
NUM_INFOGRAPHICVQA = 2_800 # InfographicVQA validation split has 2,800 samples

MAX_LENGTH = 256
HF_DATASET_REPO = "abhinav0231/lily-pretrain-alignment-dataset"  # Target HF Repo for preprocessed dataset

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
if "<image>" not in tokenizer.get_vocab():
    tokenizer.add_tokens(["<image>"], special_tokens=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print("Tokenizer and configurations initialized.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Tokenizer and configurations initialized.


## Image and Text Processing Utilities

In [4]:
def resize_image(image, max_side=448):
    """Resize image to keep max_side <= 448 to save space while preserving visual detail."""
    try:
        image = image.convert("RGB")
        image.thumbnail((max_side, max_side))
        return image
    except Exception:
        # Return dummy black image if corrupted
        return Image.new("RGB", (384, 384), (0, 0, 0))

def tokenize_sample(prompt, answer, tokenizer, max_length=MAX_LENGTH):
    """Tokenize prompt and answer to construct input_ids, attention_mask, and labels."""
    prompt_tokens = tokenizer.encode(prompt, add_special_tokens=False)
    answer_tokens = tokenizer.encode(answer, add_special_tokens=False)

    bos = [tokenizer.bos_token_id] if tokenizer.bos_token_id is not None else []
    eos = [tokenizer.eos_token_id] if tokenizer.eos_token_id is not None else []

    input_ids = bos + prompt_tokens + answer_tokens + eos
    labels = [-100] * (len(bos) + len(prompt_tokens)) + answer_tokens + eos
    attention_mask = [1] * len(input_ids)

    if len(input_ids) > max_length:
        input_ids = input_ids[:max_length]
        labels = labels[:max_length]
        attention_mask = attention_mask[:max_length]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

## Ingest and Process Datasets

In [5]:
def process_laion(n):
    print(f">> Ingesting and processing LAION (streaming split, target: {n})...")
    ds = load_dataset("lmms-lab/LLaVA-ReCap-558K", split="train", streaming=True)
    ds = ds.shuffle(seed=42, buffer_size=1000)
    
    out = []
    pbar = tqdm(total=n, desc="LAION")
    for ex in ds:
        if len(out) >= n:
            break
        try:
            img = ex["image"]
            img = resize_image(img)
            
            conversations = ex.get("conversations", [])
            caption = ""
            for msg in conversations:
                if msg.get("from") == "gpt":
                    caption = msg.get("value", "").strip()
                    break
            
            if caption:
                tokens = tokenize_sample("<image>\nDescribe this image.", caption, tokenizer)
                out.append({
                    "image": img,
                    **tokens
                })
                pbar.update(1)
        except Exception:
            continue
    pbar.close()
    print(f"   Processed {len(out)} LAION samples")
    return out

In [7]:
def process_ai2d(n):
    print(f">> Ingesting and processing AI2D (streaming split, target: {n})...")
    ds = load_dataset("lmms-lab/ai2d", split="test", streaming=True)
    
    out = []
    pbar = tqdm(desc="AI2D")
    for ex in ds:
        if len(out) >= n:
            break
        try:
            img = ex["image"]
            img = resize_image(img)
            q = ex.get("question", "Describe this diagram.").strip()
            a = ex.get("answer", "").strip()
            if a:
                tokens = tokenize_sample(f"<image>\n{q}", a, tokenizer)
                out.append({
                    "image": img,
                    **tokens
                })
                pbar.update(1)
        except Exception:
            continue
    pbar.close()
    print(f"   Processed {len(out)} AI2D samples")
    return out

In [9]:
def process_chartqa(n):
    print(f">> Ingesting and processing ChartQA (streaming split, target: {n})...")
    ds = load_dataset("HuggingFaceM4/ChartQA", split="train", streaming=True)
    
    out = []
    pbar = tqdm(total=n, desc="ChartQA")
    for ex in ds:
        if len(out) >= n:
            break
        try:
            img = ex["image"]
            img = resize_image(img)
            q = ex.get("query", ex.get("question", "Describe this chart.")).strip()
            label = ex.get("label", ex.get("answer", ""))
            a = label[0] if isinstance(label, list) and len(label) > 0 else label
            a = str(a).strip()
            if a:
                tokens = tokenize_sample(f"<image>\n{q}", a, tokenizer)
                out.append({
                    "image": img,
                    **tokens
                })
                pbar.update(1)
        except Exception:
            continue
    pbar.close()
    print(f"   Processed {len(out)} ChartQA samples")
    return out

In [8]:
def process_docvqa(n):
    print(f">> Ingesting and processing DocVQA (static validation split, target: {n})...")
    ds = load_dataset("lmms-lab/DocVQA", "DocVQA", split="validation", streaming=False)
    
    out = []
    max_range = min(n, len(ds))
    for i in tqdm(range(max_range), desc="DocVQA"):
        try:
            ex = ds[i]
            img = ex["image"]
            img = resize_image(img)
            q = ex["question"].strip()
            a = ex["answers"][0] if len(ex["answers"]) else ""
            a = str(a).strip()
            if a:
                tokens = tokenize_sample(f"<image>\n{q}", a, tokenizer)
                out.append({
                    "image": img,
                    **tokens
                })
        except Exception:
            continue
    print(f"   Processed {len(out)} DocVQA samples")
    return out

In [10]:
def process_infographicvqa(n):
    print(f">> Ingesting and processing InfographicVQA (static validation split, target: {n})...")
    ds = load_dataset("lmms-lab/DocVQA", "InfographicVQA", split="validation", streaming=False)
    
    out = []
    max_range = min(n, len(ds))
    for i in tqdm(range(max_range), desc="InfographicVQA"):
        try:
            ex = ds[i]
            img = ex["image"]
            img = resize_image(img)
            q = ex["question"].strip()
            a = ex["answers"][0] if len(ex["answers"]) else ""
            a = str(a).strip()
            if a:
                tokens = tokenize_sample(f"<image>\n{q}", a, tokenizer)
                out.append({
                    "image": img,
                    **tokens
                })
        except Exception:
            continue
    print(f"   Processed {len(out)} InfographicVQA samples")
    return out

## Build and Shuffle Mixed Dataset

In [11]:
all_samples = []
all_samples.extend(process_laion(NUM_LAION))
all_samples.extend(process_ai2d(NUM_AI2D))
all_samples.extend(process_chartqa(NUM_CHARTQA))
all_samples.extend(process_docvqa(NUM_DOCVQA))
all_samples.extend(process_infographicvqa(NUM_INFOGRAPHICVQA))

random.seed(42)
random.shuffle(all_samples)
print(f"\n>> Ingestion finished. Total preprocessed samples: {len(all_samples)}")

>> Ingesting and processing LAION (streaming split, target: 100000)...


README.md:   0%|          | 0.00/472 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

LAION:   0%|          | 0/100000 [00:00<?, ?it/s]

...

InfographicVQA/test-00003-of-00004.parqu(…):   0%|          | 0.00/81.0M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/2801 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3288 [00:00<?, ? examples/s]

InfographicVQA:   0%|          | 0/2800 [00:00<?, ?it/s]

   Processed 2800 InfographicVQA samples

>> Ingestion finished. Total preprocessed samples: 121237


## Package and Push to Hugging Face Hub

In [16]:
# Define HF Dataset Features
features = Features({
    "image": HFImage(),
    "input_ids": Sequence(Value("int32")),
    "attention_mask": Sequence(Value("int8")),
    "labels": Sequence(Value("int32")),
})

import io
from concurrent.futures import ThreadPoolExecutor

def encode_sample(sample):
    """Compress PIL Image to JPEG bytes in-place to save memory and allow instant dataset creation."""
    img = sample.get("image")
    if isinstance(img, Image.Image):
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=90)
        sample["image"] = buf.getvalue()
    return sample

print(">> Encoding images in-place using ThreadPoolExecutor...")
with ThreadPoolExecutor(max_workers=16) as executor:
    list(tqdm(executor.map(encode_sample, all_samples), total=len(all_samples), desc="Encoding images"))

print(">> Wrapping pre-encoded list into Hugging Face Dataset...")
hf_dataset = Dataset.from_list(all_samples, features=features)
print(f"Dataset structure: {hf_dataset}")

print(f">> Pushing dataset to: {HF_DATASET_REPO}...")
hf_dataset.push_to_hub(HF_DATASET_REPO, private=False)
print("\nSuccess! Preprocessed alignment dataset uploaded successfully.")

>> Encoding images in-place using ThreadPoolExecutor...


Encoding images:   0%|          | 0/121237 [00:00<?, ?it/s]

>> Wrapping pre-encoded list into Hugging Face Dataset (instantaneous)...
Dataset structure: Dataset({
    features: ['image', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 121237
})
>> Pushing dataset to: abhinav0231/lily-pretrain-alignment-dataset...


Uploading the dataset shards:   0%|          | 0/8 [00:00<?, ? shards/s]

Map:   0%|          | 0/15155 [00:00<?, ? examples/s]

...

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmpt87krjob.parquet              :   0%|          |  436kB /  462MB            


Success! Preprocessed alignment dataset uploaded successfully.


In [18]:
# --- Create and Upload Complete Text Subset ---
from datasets import Features, Image as HFImage, Value, Dataset

print(f">> Decoding all {len(all_samples)} samples back to text...")
text_samples = []

for sample in all_samples:
    # Extract prompt IDs (where labels are masked with -100)
    prompt_ids = [tid for tid, lid in zip(sample["input_ids"], sample["labels"]) if lid == -100]
    # Extract answer IDs (where labels are not -100)
    answer_ids = [lid for lid in sample["labels"] if lid != -100]
    
    # Decode back to string
    question_text = tokenizer.decode(prompt_ids, skip_special_tokens=False).strip()
    answer_text = tokenizer.decode(answer_ids, skip_special_tokens=True).strip()
    
    text_samples.append({
        "image": sample["image"], # Uses the pre-compressed bytes in memory
        "question": question_text,
        "answer": answer_text
    })

# Define Features
text_features = Features({
    "image": HFImage(),
    "question": Value("string"),
    "answer": Value("string"),
})

print(">> Packaging text dataset...")
text_dataset = Dataset.from_list(text_samples, features=text_features)

print(f">> Pushing complete text configuration to Hugging Face: {HF_DATASET_REPO} (config: 'text')...")
text_dataset.push_to_hub(HF_DATASET_REPO, config_name="text", private=False)
print("\nSuccess! Complete text subset uploaded. You can now select 'text' in the Hugging Face viewer dropdown.")

>> Decoding all 121237 samples back to text...
>> Packaging text dataset...
>> Pushing complete text configuration to Hugging Face: abhinav0231/lily-pretrain-alignment-dataset (config: 'text')...


Uploading the dataset shards:   0%|          | 0/8 [00:00<?, ? shards/s]

Map:   0%|          | 0/15155 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmp9mpqvy0u.parquet              :  22%|##2       | 98.4MB /  437MB            


Success! Complete text subset uploaded. You can now select 'text' in the Hugging Face viewer dropdown.
